In [ ]:
import pandas as pd
import ast
from neo4j import GraphDatabase
import matplotlib.pyplot as plt

# --- 1. CONFIGURAZIONE ---
URI = "bolt://localhost:7687"
AUTH = ("neo4j", "password")  # <--- INSERISCI LA TUA PASSWORD QUI

# Funzione wrapper per eseguire query (gestisce transazioni e risultati)
def run_query(query, params={}):
    with GraphDatabase.driver(URI, auth=AUTH) as driver:
        with driver.session() as session:
            result = session.run(query, params)
            return [record.data() for record in result]

def clear_db():
    """Pulisce interamente il database per ripartire da zero."""
    print("🧹 Pulizia database in corso...")
    run_query("MATCH (n) DETACH DELETE n")
    # Rimuove anche eventuali constraint vecchi per evitare conflitti
    run_query("CALL apoc.schema.assert({},{})") # Se APOC è installato, altrimenti ignora errori qui
    print("✔ Database pulito.")

# --- 2. CARICAMENTO E PULIZIA DATI ---
print("📂 Caricamento e pulizia dati...")

# A. Arxiv Papers
# Requisito PDF: "date (solo la data di pubblicazione, non l'intero timestamp)" 
df_papers = pd.read_csv('arxiv_papers.csv')

# Convertiamo la stringa "['Autore1', 'Autore2']" in una lista Python reale
df_papers['authors_list'] = df_papers['authors'].apply(ast.literal_eval)

# Pulizia Data: Da "2021-11-22T18:52:03+00:00" a "2021-11-22"
df_papers['date_clean'] = df_papers['published'].apply(lambda x: x.split('T')[0])

# B. Papers with Topics
df_topics = pd.read_csv('papers_with_topics.csv')
# Convertiamo le stringhe in liste/numeri
df_topics['top_topics_list'] = df_topics['top_topics'].apply(ast.literal_eval)
df_topics['top_probs_list'] = df_topics['top_probabilities'].apply(ast.literal_eval)

# C. Topic Definitions
df_defs = pd.read_csv('topic_definitions.csv')

print(f"✔ Dati pronti: {len(df_papers)} papers, {len(df_topics)} topic links, {len(df_defs)} definitions.")

# --- 3. CREAZIONE CONSTRAINTS (VINCOLI) ---
# Fondamentale per le performance e per evitare duplicati (MERGE più veloce)
def create_constraints():
    print("🔒 Creazione vincoli (Constraints)...")
    queries = [
        "CREATE CONSTRAINT paper_url_unique IF NOT EXISTS FOR (p:Paper) REQUIRE p.url IS UNIQUE",
        "CREATE CONSTRAINT author_name_unique IF NOT EXISTS FOR (a:Author) REQUIRE a.name IS UNIQUE",
        "CREATE CONSTRAINT topic_id_unique IF NOT EXISTS FOR (t:Topic) REQUIRE t.id IS UNIQUE"
    ]
    for q in queries:
        run_query(q)
    print("✔ Vincoli attivi.")

# --- ESECUZIONE ---
# clear_db() # Togli il commento se vuoi cancellare tutto prima di iniziare
create_constraints()

In [ ]:
# -------------------------------------------------------------------------
# 2. INGESTION DEI DATI (Batch Optimization)
# -------------------------------------------------------------------------

def ingest_data_batch(query, batch_data):
    """Esegue una query di scrittura passando un batch di dati come parametro"""
    # Nota: passiamo 'batch' come parametro alla query Cypher per efficienza
    run_query(query, {'batch': batch_data})

# --- A. INSERIMENTO TOPIC ---
# Usiamo topic_definitions.csv. Creiamo nodi Topic con ID, Nome e Keywords.
# UNWIND è un comando Cypher che 'srotola' la lista python riga per riga.
query_topics = """
UNWIND $batch AS row
MERGE (t:Topic {id: row.Topic})
SET t.name = row.Name, 
    t.keywords = row.Representation
"""
print("Inserimento Topics...")
# df_defs ha colonne: Topic, Count, Name, Representation...
ingest_data_batch(query_topics, df_defs.to_dict('records'))


# --- B. INSERIMENTO PAPER E AUTORI (Relazione :AUTHORED) ---
# Questa query è cruciale: crea il Paper e collega gli Autori in un colpo solo.
query_papers_authors = """
UNWIND $batch AS row
MERGE (p:Paper {url: row.url})
SET p.title = row.title,
    p.abstract = row.abstract,
    p.date = date(row.date)  // Convertiamo la stringa YYYY-MM-DD in Date Neo4j

WITH p, row
UNWIND row.authors_list AS author_name
MERGE (a:Author {name: trim(author_name)})
MERGE (a)-[:AUTHORED]->(p)
"""

print("Inserimento Papers e Autori (Batching)...")
batch_size = 1000
records_papers = df_papers.to_dict('records')

# Eseguiamo a blocchi di 1000 per non sovraccaricare la memoria
for i in range(0, len(records_papers), batch_size):
    batch = records_papers[i:i+batch_size]
    ingest_data_batch(query_papers_authors, batch)
    print(f" - Caricati {i + len(batch)} / {len(records_papers)} papers")


# --- C. INSERIMENTO RELAZIONE PAPER-TOPIC (:ON_TOPIC) ---
# Usiamo papers_with_topics.csv. Colleghiamo ogni Paper ai suoi top 3 topic.
query_topic_rels = """
UNWIND $batch AS row
MATCH (p:Paper {url: row.url})
// Iteriamo sugli indici della lista (0, 1, 2)
FOREACH (i IN range(0, size(row.top_topics_list)-1) | 
    MERGE (t:Topic {id: row.top_topics_list[i]})
    MERGE (p)-[r:ON_TOPIC]->(t)
    // Impostiamo la probabilità sulla relazione (richiesta per Jaccard pesata/Ruzicka)
    SET r.probability = row.top_probs_list[i]
)
"""

print("Inserimento Relazioni Paper-Topic...")
records_topics = df_topics.to_dict('records')
for i in range(0, len(records_topics), batch_size):
    batch = records_topics[i:i+batch_size]
    ingest_data_batch(query_topic_rels, batch)
    print(f" - Collegate {i + len(batch)} relazioni topic")

print("✔ Ingestion base completata.")

In [ ]:
# -------------------------------------------------------------------------
# 2. INGESTION DEI DATI (Sequenza da Homework)
# -------------------------------------------------------------------------

def ingest_data_batch(query, batch_data):
    """Esegue una query di scrittura passando un batch di dati."""
    run_query(query, {'batch': batch_data})

batch_size = 1000

# --- STEP 1: Creazione Nodi PAPER ---
# Proprietà: title, abstract, date (YYYY-MM-DD), url
print("1. Creazione Nodi Paper...")
query_papers = """
UNWIND $batch AS row
MERGE (p:Paper {url: row.url})
SET p.title = row.title,
    p.abstract = row.abstract,
    p.date = date(row.date)  // Conversione stringa -> Oggetto Date
"""
records_papers = df_papers.to_dict('records')
for i in range(0, len(records_papers), batch_size):
    ingest_data_batch(query_papers, records_papers[i:i+batch_size])
print(f"   ✔ Completato per {len(records_papers)} articoli.")


# --- STEP 2: Creazione Nodi AUTHOR ---
# Proprietà: name
print("2. Creazione Nodi Author...")
query_authors = """
UNWIND $batch AS row
UNWIND row.authors_list AS author_name
MERGE (a:Author {name: trim(author_name)})
"""
# Riusiamo records_papers perché contiene la lista autori
for i in range(0, len(records_papers), batch_size):
    ingest_data_batch(query_authors, records_papers[i:i+batch_size])
print("   ✔ Completato.")


# --- STEP 3: Creazione Nodi TOPIC ---
# Estratti dagli abstract (usiamo topic_definitions.csv)
print("3. Creazione Nodi Topic...")
query_topics = """
UNWIND $batch AS row
MERGE (t:Topic {id: row.Topic})
SET t.name = row.Name, 
    t.keywords = row.Representation
"""
ingest_data_batch(query_topics, df_defs.to_dict('records'))
print("   ✔ Completato.")


# --- STEP 4: Creazione RELAZIONI ---

# 4a. Relazione :AUTHORED (Paper <-> Author)
print("4a. Creazione relazione :AUTHORED...")
query_rel_authored = """
UNWIND $batch AS row
MATCH (p:Paper {url: row.url})
UNWIND row.authors_list AS author_name
MATCH (a:Author {name: trim(author_name)})
MERGE (a)-[:AUTHORED]->(p)
"""
for i in range(0, len(records_papers), batch_size):
    ingest_data_batch(query_rel_authored, records_papers[i:i+batch_size])


# 4b. Relazione :ON_TOPIC (Paper <-> Topic) con probabilità
print("4b. Creazione relazione :ON_TOPIC...")
query_rel_ontopic = """
UNWIND $batch AS row
MATCH (p:Paper {url: row.url})
// Iteriamo sugli indici della lista topics (0, 1, 2)
FOREACH (i IN range(0, size(row.top_topics_list)-1) | 
    MERGE (t:Topic {id: row.top_topics_list[i]})
    MERGE (p)-[r:ON_TOPIC]->(t)
    SET r.probability = row.top_probs_list[i]
)
"""
records_topics = df_topics.to_dict('records')
for i in range(0, len(records_topics), batch_size):
    ingest_data_batch(query_rel_ontopic, records_topics[i:i+batch_size])


# 4c. Relazione :COAUTHORED_WITH (Author <-> Author)
# Calcolata direttamente nel grafo
print("4c. Creazione relazione :COAUTHORED_WITH (Batch)...")
query_coauthor = """
MATCH (a1:Author)
CALL (a1) {
    MATCH (a1)-[:AUTHORED]->(p:Paper)<-[:AUTHORED]-(a2:Author)
    WHERE elementId(a1) < elementId(a2)
    WITH a1, a2, count(p) as common
    MERGE (a1)-[r:COAUTHORED_WITH]-(a2)
    SET r.count = common
} IN TRANSACTIONS OF 100 ROWS
"""
run_query(query_coauthor)

print("\n✔ Tutti i nodi e relazioni (Blocco 2) sono stati caricati correttamente.")

In [ ]:
# -------------------------------------------------------------------------
# 5. CALCOLO SIMILARITÀ JACCARD PESATA (RUZICKA)
# -------------------------------------------------------------------------
# Implementazione diretta in Cypher come da best-practice (image_da872e.jpg)
# ma adattata alla formula richiesta (Min/Max).

def calculate_ruzicka_similarity():
    print("🧠 Avvio calcolo Similarità Ruzicka (Server-side)...")

    # PASSO A: Pre-calcolo dei pesi totali per ogni Paper
    # Questo serve per il denominatore della formula.
    # Salva la somma delle probabilità dei topic come proprietà 'topic_weight' sul nodo Paper.
    query_weights = """
    MATCH (p:Paper)-[r:ON_TOPIC]->()
    WITH p, sum(r.probability) AS total_weight
    SET p.topic_weight = total_weight
    """
    print(" - Fase 1: Calcolo pesi totali (topic_weight)...")
    run_query(query_weights)

    # PASSO B: Calcolo Similarità e Creazione Relazioni
    # Confronta coppie di paper che condividono almeno un topic.
    query_similarity = """
    MATCH (p1:Paper)-[r1:ON_TOPIC]->(t:Topic)<-[r2:ON_TOPIC]-(p2:Paper)
    WHERE elementId(p1) < elementId(p2)  // Evita duplicati (A-B vs B-A) e self-loops
    
    // 1. Calcolo Numeratore: Somma dei Minimi (Intersezione)
    WITH p1, p2, sum(
        CASE WHEN r1.probability < r2.probability THEN r1.probability 
        ELSE r2.probability END
    ) AS intersection_sum
    
    // 2. Calcolo Denominatore: Somma dei Massimi (Unione)
    // Formula: Union = Peso1 + Peso2 - Intersezione
    WITH p1, p2, intersection_sum, 
         (p1.topic_weight + p2.topic_weight - intersection_sum) AS union_sum
    
    // 3. Calcolo Score e Filtro
    WHERE union_sum > 0
    WITH p1, p2, intersection_sum / union_sum AS sim_score
    WHERE sim_score > 0.15  // SOGLIA: Salviamo solo se > 0.15 per non riempire il DB di archi deboli
    
    // 4. Scrittura Relazione
    MERGE (p1)-[s:SIMILAR_TOPIC]-(p2)
    SET s.score = sim_score
    """
    
    print(" - Fase 2: Creazione relazioni :SIMILAR_TOPIC (può richiedere qualche minuto)...")
    # Nota: Senza APOC, questa query è un'unica transazione. 
    # Su 8000 nodi è gestibile, ma se dovesse dare errore di memoria, 
    # useremo la tecnica 'CALL ... IN TRANSACTIONS' anche qui.
    run_query(query_similarity)
    print("✔ Calcolo similarità completato.")

# Esecuzione
calculate_ruzicka_similarity()

In [ ]:
# -------------------------------------------------------------------------
# 4. ANALISI E QUERY RICHIESTE (HOMEWORK PARTE 2)
# -------------------------------------------------------------------------

# --- QUERY 1 & 4 & 6: Istruzioni per Neo4j Desktop ---
print("\n" + "="*50)
print("ISTRUZIONI PER SCREENSHOT (NEO4J DESKTOP)")
print("="*50)
print("Q1 (Schema): Copia ed esegui: CALL db.schema.visualization()")
print("Q4 (Shortest Path): Scegli due nomi dalla Q2 qui sotto ed esegui:")
print("   MATCH (a1:Author {name: 'Nome1'}), (a2:Author {name: 'Nome2'})")
print("   MATCH p = shortestPath((a1)-[:COAUTHORED_WITH*]-(a2)) RETURN p")
print("Q6 (Co-Authorship Graph): Copia ed esegui:")
print("   MATCH (a:Author) WITH a, count {(a)-[:AUTHORED]->()} as cnt ORDER BY cnt DESC LIMIT 20")
print("   WITH collect(a) as tops MATCH (a1)-[r:COAUTHORED_WITH]-(a2)")
print("   WHERE a1 IN tops AND a2 IN tops RETURN a1, r, a2")
print("="*50 + "\n")

# --- QUERY 2: Top 5 Autori ---
print("--- Q2: Top 5 Autori ---")
q2 = """
MATCH (a:Author)-[:AUTHORED]->(p:Paper)
RETURN a.name as Author, count(p) as PaperCount
ORDER BY PaperCount DESC LIMIT 5
"""
df_authors = pd.DataFrame(run_query(q2))
print(df_authors)

# --- QUERY 3: Andamento Temporale (Mensile) ---
# Usiamo il nodo Date creato nel Blocco 2
print("\n--- Q3: Generazione Grafico Temporale ---")
q3 = """
MATCH (p:Paper)
RETURN substring(toString(p.date), 0, 7) AS YearMonth, count(p) as Count
ORDER BY YearMonth
"""
df_trend = pd.DataFrame(run_query(q3))

if not df_trend.empty:
    plt.figure(figsize=(10,6))
    plt.bar(df_trend['YearMonth'], df_trend['Count'], color='skyblue', edgecolor='navy')
    plt.title("Pubblicazioni per Mese (2021)")
    plt.xticks(rotation=45)
    plt.ylabel("Numero Paper")
    plt.tight_layout()
    plt.show()
else:
    print("Nessun dato temporale trovato.")

# --- QUERY 5: Similarità Media tra Autori ---
# Formula complessa: 1/(|P1|*|P2|) * Sum(Sim_Paper_Pairs)
print("\n--- Q5: Calcolo Similarità tra i primi 2 Top Autori ---")
if len(df_authors) >= 2:
    a1 = df_authors.iloc[0]['Author']
    a2 = df_authors.iloc[1]['Author']
    
    q5 = """
    MATCH (a1:Author {name: $name1})-[:AUTHORED]->(p1:Paper)
    MATCH (a2:Author {name: $name2})-[:AUTHORED]->(p2:Paper)
    // Cerchiamo relazioni di similarità pre-calcolate tra i loro paper
    OPTIONAL MATCH (p1)-[s:SIMILAR_TOPIC]-(p2)
    WITH count(DISTINCT p1) as lenP1, count(DISTINCT p2) as lenP2, sum(coalesce(s.score, 0)) as totalSim
    // Formula finale
    RETURN (1.0 / (lenP1 * lenP2)) * totalSim as AuthorSimilarity
    """
    res = run_query(q5, {'name1': a1, 'name2': a2})
    score = res[0]['AuthorSimilarity'] if res else 0
    print(f"Similarità tra {a1} e {a2}: {score:.4f}")
else:
    print("Non abbastanza autori per il test Q5.")

# Chiusura driver alla fine di tutto
driver.close()
print("\n✔ Notebook completato. Driver chiuso.")

---
## Riepilogo Query Cypher per Neo4j Browser

### 1. Visualizzare lo schema del database
```cypher
CALL db.schema.visualization()
```

### 2. Top 5 autori per numero di articoli
```cypher
MATCH (a:Author)-[:AUTHORED]->(p:Paper)
RETURN a.name AS author, COUNT(p) AS papers_count
ORDER BY papers_count DESC
LIMIT 5
```

### 3. Andamento temporale delle pubblicazioni
```cypher
MATCH (p:Paper)
RETURN substring(p.date, 0, 4) AS year, COUNT(p) AS papers_count
ORDER BY year
```

### 4. Percorso più breve tra due autori
```cypher
MATCH path = shortestPath(
    (a1:Author {name: 'AUTORE1'})-[:AUTHORED|COAUTHORED_WITH*]-(a2:Author {name: 'AUTORE2'})
)
RETURN path
```

### 5. Rete di co-authorship dei Top 20 autori
```cypher
MATCH (a:Author)-[:AUTHORED]->(p:Paper)
WITH a, COUNT(p) AS papers_count
ORDER BY papers_count DESC
LIMIT 20
WITH COLLECT(a) AS top_authors
UNWIND top_authors AS a1
UNWIND top_authors AS a2
MATCH (a1)-[r:COAUTHORED_WITH]-(a2)
WHERE id(a1) < id(a2)
RETURN a1, r, a2
```